In [1]:
!pip install -q "Pillow>=10.2.0,<12.0"
!pip install -q transformers>=4.40 "bitsandbytes>=0.46.1" "accelerate>=0.25" datasets matplotlib numpy pandas tqdm scipy

In [ ]:
import os
from huggingface_hub import login

# Set HF_TOKEN in your shell environment (e.g. export HF_TOKEN=hf_...) or paste when prompted.
login(token=os.environ.get('HF_TOKEN'))


In [3]:
import importlib.util
spec = importlib.util.find_spec("ner_filter")
print("ner_filter found:", spec is not None, "| path:", getattr(spec, "origin", "N/A"))

ner_filter found: False | path: N/A


In [ ]:
import os
import sys
from pathlib import Path

# The notebook lives in the gemma/ directory — add it to sys.path for direct imports.
_gemma_dir = str(Path(os.getcwd()))
if _gemma_dir not in sys.path:
    sys.path.insert(0, _gemma_dir)
print('Gemma dir:', _gemma_dir)
print(os.listdir(_gemma_dir))


['dataset.py', 'evaluation.py', 'model_utils.py', 'ner_filter.py', 'run.py', 'requirements.txt', 'visualization.py', 'saliency', 'config.py', 'explainability_gemma.ipynb', '__pycache__']


In [ ]:
from config import Config
from pathlib import Path

# ── Dataset toggle ─────────────────────────────────────────────────────
USE_COCO = False   # True → COCO (MS-COCO 2017 captions)  |  False → ROCO v2
# ────────────────────────────────────────────────────────────────────────────

cfg = Config()
cfg.model_id          = 'google/gemma-3-4b-it'
cfg.load_in_4bit      = True
cfg.attn_implementation = 'eager'

# Results are stored under gemma/results_*/
base_results_dir = Path.cwd()

if USE_COCO:
    cfg.dataset_name   = 'lmms-lab/COCO-Caption'
    cfg.dataset_config = ''
    cfg.dataset_split  = 'val'
    cfg.image_column   = 'image'
    cfg.caption_column = 'answer'
    cfg.output_dir     = str(base_results_dir / 'results_coco')
else:
    cfg.dataset_name   = 'eltorio/ROCOv2-radiology'
    cfg.dataset_config = ''
    cfg.dataset_split  = 'train'
    cfg.image_column   = 'image'
    cfg.caption_column = 'caption'
    cfg.output_dir     = str(base_results_dir / 'results_roco')

cfg.num_samples       = 15
cfg.max_new_tokens    = 100
cfg.methods           = ['gradcam', 'attention', 'gmar_l1', 'gmar_l2']
cfg.mask_ratios       = [0.1, 0.2, 0.4, 0.3, 0.5]
cfg.save_visualizations = True

# NER filtering config
cfg.use_ner_filter     = False
cfg.ner_model          = 'Clinical-AI-Apollo/Medical-NER'
cfg.ner_entity_groups  = [
    'DISEASE_DISORDER', 'SIGN_SYMPTOM'
]

EVAL_MODE    = 'per_token'
CONTENT_ONLY = True

print(f"[config] Model:   {cfg.model_id}")
print(f"[config] Dataset: {'COCO (lmms-lab/COCO-Caption val)' if USE_COCO else 'ROCO v2 (eltorio/ROCOv2-radiology)'}")
print(f"[config] Output dir: {cfg.output_dir}")


[config] Model:   google/gemma-3-4b-it
[config] Dataset: COCO (lmms-lab/COCO-Caption val)
[config] Output dir: results_coco


In [7]:
from model_utils import load_model_and_processor
model, processor = load_model_and_processor(cfg)

[model] Loading google/gemma-3-4b-it …


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

[model] Loaded.  device_map = n/a


In [8]:
# deberta model for prompt classification
from transformers import pipeline
pipe = pipeline("token-classification", model="Clinical-AI-Apollo/Medical-NER", aggregation_strategy='simple')

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

In [9]:
from dataset import load_dataset_samples
samples = load_dataset_samples(cfg)

[dataset] Loading 'lmms-lab/COCO-Caption' split='val' (streaming, 15 samples) …


Loading samples: 100%|██████████| 15/15 [00:00<00:00, 30.30it/s]

[dataset] Loaded 15 samples.


In [10]:
print(cfg.prompt)
cfg.prompt = "Write a single-sentence caption for this image. "

Write a single-sentence radiology caption for this medical image. Just write the caption, do not prefix by saying 'Here is a caption' just write it.


In [ ]:
import gc
import json
import time
import torch
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm as tqdm_nb

from model_utils import (
    generate_caption, get_tokenizer, get_image_token_positions,
    get_token_probabilities, get_content_token_mask,
    build_tf_inputs, move_inputs_to_device,
)
from ner_filter import build_variants
from saliency import get_saliency_fn
from evaluation import (
    evaluate_faithfulness_average,
    evaluate_faithfulness_per_token,
    evaluate_faithfulness_random,
)
from visualization import (
    save_token_saliency_grid, save_comparison_figure,
    save_perturbation_curve, save_aggregate_curves,
)

out_dir = Path(cfg.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)

with open(out_dir / 'config.json', 'w') as f:
    json.dump(vars(cfg), f, indent=2, default=str)

tok = get_tokenizer(processor)
all_sample_results = []

# -----------------------------------------------------------------------------
# Main loop
# -----------------------------------------------------------------------------
t_total = time.time()

for i in tqdm_nb(range(len(samples)), desc='Processing samples'):
    sample      = samples[i]
    image       = sample['image']
    ref_caption = sample.get('caption', '')
    sample_id   = sample.get('id', str(i))

    # 1. Generate caption
    gen_ids, gen_text, input_len, inputs = generate_caption(model, processor, image, cfg)
    total_len = gen_ids.shape[1]
    num_gen   = total_len - input_len
    print(f'\nSample {i}: generated {num_gen} tokens - {gen_text[:100]}')
    if num_gen == 0:
        continue

    # 2. Image-token positions
    img_positions = get_image_token_positions(inputs)

    # 3. Teacher-forcing inputs
    tf_inputs = build_tf_inputs(inputs, gen_ids, input_len)

    # 4. Original token probabilities
    orig_probs = get_token_probabilities(model, tf_inputs, gen_ids, input_len)

    # 5. Token strings & content mask
    token_strings = {}
    for pos in range(input_len, total_len):
        tid = gen_ids[0, pos].item()
        token_strings[pos] = tok.decode([tid], skip_special_tokens=True).strip()
    content_mask = get_content_token_mask(tok, gen_ids, input_len)
    content_positions = [
        pos for pos, keep in zip(range(input_len, total_len), content_mask) if keep
    ]

    # 5b. NER variants
    variants = build_variants(gen_text, gen_ids, input_len, tok, cfg)

    # 6. Saliency + evaluation per method, per variant
    sample_dir = out_dir / f'sample_{i:04d}'
    sample_dir.mkdir(parents=True, exist_ok=True)

    all_saliency        = {}
    method_eval_results = {}

    for method in cfg.methods:
        print(f'  {method} ...')
        compute  = get_saliency_fn(method)
        sal_maps = compute(model, tf_inputs, gen_ids, input_len, img_positions, cfg)
        all_saliency[method] = sal_maps

        for var_name, var_info in variants.items():
            var_positions = var_info['token_positions']
            var_content_mask = [
                (content_mask[pos - input_len] if CONTENT_ONLY else True)
                and (pos in var_positions)
                for pos in range(input_len, total_len)
            ]
            var_sal_maps = {p: s for p, s in sal_maps.items() if p in var_positions}
            if not var_sal_maps:
                continue

            suffix     = f'_{var_name}' if var_name != 'original' else ''
            result_key = f'{method}{suffix}'

            if EVAL_MODE == 'average':
                ev = evaluate_faithfulness_average(
                    model, inputs, gen_ids, input_len, var_sal_maps, orig_probs, cfg)
            else:
                ev = evaluate_faithfulness_per_token(
                    model, inputs, gen_ids, input_len, var_sal_maps, orig_probs, cfg,
                    content_mask=var_content_mask)

            for row in ev.get('per_token', []):
                row['token_text'] = token_strings.get(row.get('position'), '')

            print(f'    [{var_name}] AOPC = {ev["aopc"]:.4f}')
            method_eval_results[result_key] = ev

        if cfg.save_visualizations:
            save_token_saliency_grid(
                image, sal_maps, token_strings, method,
                str(sample_dir / f'saliency_{method}.png'),
                content_positions=content_positions)

    # 7. Random baseline
    random_positions = content_positions if CONTENT_ONLY else list(range(input_len, total_len))
    random_sal_maps = {
        pos: np.random.rand(cfg.image_token_grid, cfg.image_token_grid).astype(np.float32)
        for pos in random_positions
    }
    if random_sal_maps:
        rand_content_mask = [pos in random_positions for pos in range(input_len, total_len)]
        rand_ev = evaluate_faithfulness_per_token(
            model, inputs, gen_ids, input_len, random_sal_maps, orig_probs, cfg,
            content_mask=rand_content_mask)
        for row in rand_ev.get('per_token', []):
            row['token_text'] = token_strings.get(row.get('position'), '')
        print(f'  [random] AOPC = {rand_ev["aopc"]:.4f}')
        method_eval_results['random'] = rand_ev
        all_saliency['random'] = random_sal_maps

    # 8. Comparison & curve figures
    if cfg.save_visualizations:
        if len(all_saliency) >= 2:
            save_comparison_figure(
                image, all_saliency, token_strings,
                str(sample_dir / 'comparison.png'),
                content_positions=content_positions)
        save_perturbation_curve(
            method_eval_results, str(sample_dir / 'perturbation_curve.png'),
            title=f'Sample {i}')
    image.save(str(sample_dir / 'original.png'))

    # 9. Collect result
    res = {
        'sample_id':            sample_id,
        'sample_idx':           i,
        'generated_text':       gen_text,
        'reference_caption':    ref_caption,
        'num_generated_tokens': num_gen,
        'num_content_tokens':   sum(content_mask),
        'ner_variants':         {k: v['text'] for k, v in variants.items()},
        'eval': {
            m: {
                'aopc':                r.get('aopc', 0.0),
                'mean_drops_by_ratio': r.get('mean_drops_by_ratio', {}),
                'per_token':           r.get('per_token', []),
            }
            for m, r in method_eval_results.items()
        },
    }
    all_sample_results.append(res)

    with open(sample_dir / 'result.json', 'w') as f:
        json.dump(res, f, indent=2, default=str)

    # Free memory
    del tf_inputs, orig_probs, sal_maps
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

# -----------------------------------------------------------------------------
# Save final outputs
# -----------------------------------------------------------------------------
with open(out_dir / 'all_results.json', 'w') as f:
    json.dump(all_sample_results, f, indent=2, default=str)

del_rows = [
    {**{'sample_idx': r['sample_idx'], 'sample_id': r['sample_id'], 'method': m}, **tok_row}
    for r in all_sample_results
    for m, ev in r.get('eval', {}).items()
    for tok_row in ev.get('per_token', [])
]
if del_rows:
    pd.DataFrame(del_rows).to_csv(out_dir / 'per_token_drops.csv', index=False)
    with open(out_dir / 'per_token_drops.jsonl', 'w') as f:
        for row in del_rows:
            f.write(json.dumps(row, default=str) + '\n')

all_eval_by_method = {}
for res in all_sample_results:
    for m, ev in res.get('eval', {}).items():
        all_eval_by_method.setdefault(m, []).append(ev)

summary_rows = []
for method, evals in all_eval_by_method.items():
    aopc_vals = [e.get('aopc', 0.0) for e in evals]
    summary_rows.append({'method': method, 'mean_aopc': float(np.mean(aopc_vals)), 'std_aopc': float(np.std(aopc_vals)), 'n_samples': len(aopc_vals)})
if summary_rows:
    pd.DataFrame(summary_rows).to_csv(out_dir / 'summary.csv', index=False)

if cfg.save_visualizations and all_eval_by_method:
    save_aggregate_curves(all_eval_by_method, str(out_dir / 'aggregate_perturbation.png'))

elapsed = time.time() - t_total
print(f'\n{"="*60}')
print(f'Done. {len(all_sample_results)} samples in {elapsed:.0f}s')
print(f'Output: {out_dir}')
print(f'{"="*60}')


Processing samples:   0%|          | 0/15 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Sample 0: generated 12 tokens - Here are a few single-sentence captions for the image:
  gradcam ...


    [original] AOPC = -0.0028
  attention ...


    [original] AOPC = -0.0022
  gmar_l1 ...


...

    [original] AOPC = 0.0000
  gmar_l1 ...


    [original] AOPC = 0.0000
  gmar_l2 ...


    [original] AOPC = 0.0000



Sample 3: generated 12 tokens - Here's a single-sentence caption for the image:
  gradcam ...


KeyboardInterrupt: 

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

out_dir = Path(cfg.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)

# Re-build eval index from in-memory results
all_eval_by_method = {}
for res in all_sample_results:
    for m, ev in res.get('eval', {}).items():
        all_eval_by_method.setdefault(m, []).append(ev)

summary_rows = []
for method, evals in all_eval_by_method.items():
    if not evals:
        continue
    aopc_vals = [e.get('aopc', 0.0) for e in evals if isinstance(e, dict)]
    mean_aopc = float(np.mean(aopc_vals))
    std_aopc  = float(np.std(aopc_vals))
    print(f'{method:>12s}:  AOPC = {mean_aopc:.4f} +/- {std_aopc:.4f}')
    summary_rows.append({
        'method': method, 'mean_aopc': mean_aopc,
        'std_aopc': std_aopc, 'n_samples': len(aopc_vals),
    })

if summary_rows:
    df = pd.DataFrame(summary_rows)
    df.to_csv(out_dir / 'summary.csv', index=False)
    display(df)

with open(out_dir / 'all_results.json', 'w') as f:
    json.dump(all_sample_results, f, indent=2, default=str)

del_rows = [
    {**{'sample_idx': r['sample_idx'], 'sample_id': r['sample_id'], 'method': m}, **tok_row}
    for r in all_sample_results
    for m, ev in r.get('eval', {}).items()
    for tok_row in ev.get('per_token', [])
]
if del_rows:
    pd.DataFrame(del_rows).to_csv(out_dir / 'per_token_drops.csv', index=False)
    with open(out_dir / 'per_token_drops.jsonl', 'w') as f:
        for row in del_rows:
            f.write(json.dumps(row, default=str) + '\n')
    print(f'Per-token rows: {len(del_rows)}')

if cfg.save_visualizations and any(all_eval_by_method.values()):
    save_aggregate_curves(all_eval_by_method, str(out_dir / 'aggregate_perturbation.png'))

print('Results saved to', out_dir)


     gradcam:  AOPC = 0.1140 +/- 0.0781
   attention:  AOPC = 0.1358 +/- 0.0615
     gmar_l1:  AOPC = 0.1168 +/- 0.0875
     gmar_l2:  AOPC = 0.1264 +/- 0.0893
      random:  AOPC = 0.0865 +/- 0.0572


,method,mean_aopc,std_aopc,n_samples
0,gradcam,0.114002,0.078127,15
1,attention,0.135790,0.061462,15
2,gmar_l1,0.116750,0.087472,15
3,gmar_l2,0.126425,0.089274,15
4,random,0.086461,0.057183,15


Results saved to /content/drive/Othercomputers/My Mac/Thesis/results_gemmarocco


In [ ]:
print(cfg.output_dir)

results


In [ ]:
import shutil
from pathlib import Path

out_dir = Path(cfg.output_dir)
zip_path = out_dir.parent / f'{out_dir.name}_archive'
shutil.make_archive(str(zip_path), 'zip', str(out_dir))
print(f'Archive created: {zip_path}.zip  (source: {out_dir})')
